### MiddleWare
Think of it as a layer in the middle that can:

* Check inputs before they reach the LLM
* Modify prompts
* Log requests and responses
* Handle authentication
* Add guardrails/safety checks
* Track token usage and costs
* Retry failed requests

In [55]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("groq_api_key")
os.environ["GOOGLE_API_KEY"]=os.getenv("google_api_key")


### Summarization MiddleWare
- Automatically summarizes long 
conversation history.
- Reduces token usage and costs.
- Prevents context window overflow.
- Preserves important context from earlier messages.
- Improves performance in long-running conversations.


In [56]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


# Messagebased summerization 
from langchain.chat_models import init_chat_model

primary_model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

fallback_model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

model = primary_model.with_fallbacks([fallback_model])

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
    )


In [57]:
# Run with thread id
config={"configurable":{"thread_id":"test-1"}}


In [58]:
from langchain_core.messages import HumanMessage
questions=[
    "what is 2+2?",
    "what is 10*8?",
    "what is 10>3?",
    "What is 1*6?",
    "what is 6*5?",
    "what are you doing with above questions ?"
]

for q in questions:
    respose=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {respose}")
    print(f"Messages {len(respose['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='4c5e37e2-9a8e-46a9-ab6e-bf8308fbf742'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.01169132, 'completion_tokens_details': None, 'prompt_time': 0.005718892, 'prompt_tokens_details': None, 'queue_time': 0.317862175, 'total_time': 0.017410212}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8d27-5428-7b90-b157-c7c0a3501613-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages 2
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='4c5e37e2-9a8e-46a9-ab6e-bf8308fbf742'), AIMessag

### Token Size

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# Primary Model (Groq)
groq_model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# Fallback Model (Gemini)
gemini_model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"""
Hotels in {city}

1. Grand Palace Hotel
   Budget: ₹3,500/night
   Rating: 4.5/5

2. City Comfort Inn
   Budget: ₹2,200/night
   Rating: 4.2/5

3. Luxury Stay Resort
   Budget: ₹6,800/night
   Rating: 4.8/5
"""

# Primary Agent
groq_agent = create_agent(
    model=groq_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=groq_model,
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

# Fallback Agent
gemini_agent = create_agent(
    model=gemini_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=gemini_model,
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

config = {
    "configurable": {
        "thread_id": "test-1"
    }
}
#Token counter (approximate) 
def count_tokens(messages): 
    total_char=sum(len(str(m.content))for m in messages) 
    return total_char // 4 # 4 chars = 1 token

cities = ["Paris", "London"]

for city in cities:

    try:
        print(f"\nUsing Groq for {city}")

        response = groq_agent.invoke(
            {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
            config=config
        )

    except Exception as e:

        print("Groq failed.")
        print("Switching to Gemini...")
        print(e)

        response = gemini_agent.invoke(
            {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
            config=config
        )
    tokens=count_tokens(respose["messages"])
    print(f"{city}:~{tokens} tokens, {len(respose['messages'])} messages") 
    print(response["messages"][-1].content)
    


Using Groq for Paris
Groq failed.
Switching to Gemini...
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khqzc4x1e1mbf1602rwxk5vg` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99998, Requested 535. Please try again in 7m40.511999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 53.560931165s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

### Fraction

In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# -----------------------------
# Models
# -----------------------------

# Primary Model (Groq)
groq_model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# Fallback Model (Gemini)
gemini_model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

# -----------------------------
# Tool
# -----------------------------

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city."""

    return f"""
Hotels in {city}

1. Grand Palace Hotel
   Budget: ₹3,500/night
   Rating: 4.5/5
   Review: Clean rooms, friendly staff, and excellent breakfast.

2. City Comfort Inn
   Budget: ₹2,200/night
   Rating: 4.2/5
   Review: Good value for money with a convenient city-center location.

3. Luxury Stay Resort
   Budget: ₹6,800/night
   Rating: 4.8/5
   Review: Spacious rooms, premium amenities, and outstanding service.
"""

# -----------------------------
# Primary Agent (Groq)
# -----------------------------

groq_agent = create_agent(
    model=groq_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=groq_model,
            trigger=("fraction", 0.005),   #0.5% = ~640 tokens
            keep=("fraction", 0.002)      #0.2% = ~256 tokens
        )
    ]
)

# -----------------------------
# Fallback Agent (Gemini)
# -----------------------------

gemini_agent = create_agent(
    model=gemini_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=gemini_model,
            trigger=("fraction", 0.005),  # 0.5%
            keep=("fraction", 0.002)      # 0.2%
        )
    ]
)

# -----------------------------
# Config
# -----------------------------

config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

# -----------------------------
# Approx Token Counter
# -----------------------------

def count_tokens(messages):
    total_chars = sum(
        len(str(message.content))
        for message in messages
    )
    return total_chars // 4

# -----------------------------
# Test Queries
# -----------------------------

cities = [
    "Paris",
    "London",
    "New York",
    "Singapore",
    "Dubai"
]

for city in cities:

    try:
        print(f"\nUsing Groq for {city}")

        response = groq_agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=f"Find hotels in {city}"
                    )
                ]
            },
            config=config
        )

    except Exception as e:

        print("\nGroq failed.")
        print("Switching to Gemini...")
        print(f"Error: {e}")

        try:
            response = gemini_agent.invoke(
                {
                    "messages": [
                        HumanMessage(
                            content=f"Find hotels in {city}"
                        )
                    ]
                },
                config=config
            )

        except Exception as e2:

            print("\nGemini also failed.")
            print(f"Error: {e2}")
            continue

    tokens = count_tokens(response["messages"])

    print(
        f"{city}: ~{tokens} tokens, "
        f"{len(response['messages'])} messages"
    )

    print("\nLast Message:")
    print(response["messages"][-1].content)

    print("\n" + "=" * 60)


Using Groq for Paris

Groq failed.
Switching to Gemini...
Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khqzc4x1e1mbf1602rwxk5vg` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99874, Requested 231. Please try again in 1m30.72s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Gemini also failed.
Error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\n

In [50]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware,ModelFallbackMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


# Messagebased summerization 
from langchain.chat_models import init_chat_model

primary_model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

fallback_model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

model = primary_model.with_fallbacks([fallback_model])

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelFallbackMiddleware(
            groq_model,
            fallback_model,)
    ]
           
    )

# Run with thread id
config={"configurable":{"thread_id":"test-1"}}

from langchain_core.messages import HumanMessage
questions=[
    "what is 2+2?",
    "what is 10*8?",
    "what is 10>3?",
    "What is 1*6?",
    "what is 6*5?",
    "what are you doing with above questions ?"
]

for q in questions:
    respose=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {respose}")
    print(f"Messages {len(respose['messages'])}")



Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='839db1c7-dfce-4217-b405-22dc33812a0a'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.011800836, 'completion_tokens_details': None, 'prompt_time': 0.004762185, 'prompt_tokens_details': None, 'queue_time': 0.162586564, 'total_time': 0.016563021}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8d06-aa26-7783-ac39-ac372d0d846c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages 2
Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='839db1c7-dfce-4217-b405-22dc33812a0a'), AIMessa

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 26.178669612s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [52]:
from langchain.agents.middleware import ModelFallbackMiddleware

help(ModelFallbackMiddleware)

Help on class ModelFallbackMiddleware in module langchain.agents.middleware.model_fallback:

class ModelFallbackMiddleware(langchain.agents.middleware.types.AgentMiddleware)
 |  ModelFallbackMiddleware(first_model: 'str | BaseChatModel', *additional_models: 'str | BaseChatModel') -> 'None'
 |  
 |  Automatic fallback to alternative models on errors.
 |  
 |  Retries failed model calls with alternative models in sequence until
 |  success or all models exhausted. Primary model specified in `create_agent`.
 |  
 |  Example:
 |      ```python
 |      from langchain.agents.middleware import ModelFallbackMiddleware
 |      from langchain.agents import create_agent
 |  
 |      fallback = ModelFallbackMiddleware(
 |          "openai:gpt-5.5",  # Try first on error
 |          "anthropic:claude-sonnet-4-5-20250929",  # Then this
 |      )
 |  
 |      agent = create_agent(
 |          model="openai:gpt-5.5",  # Primary model
 |          middleware=[fallback],
 |      )
 |  
 |      # If prima

In [53]:
import inspect
from langchain.agents.middleware import ModelFallbackMiddleware

print(inspect.signature(ModelFallbackMiddleware))

(first_model: 'str | BaseChatModel', *additional_models: 'str | BaseChatModel') -> 'None'


### Human in the loop MiddleWare

In [64]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# Model
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# -------------------------
# Tools
# -------------------------

@tool
def draft_email(topic: str) -> str:
    """Create draft email."""

    return f"""
Subject: Meeting Request

Hello Team,

I would like to schedule a meeting regarding {topic}.

Thanks,
Dhruvi
"""

@tool
def edit_email(email: str, feedback: str) -> str:
    """Edit email using human feedback."""

    prompt = f"""
You are an email editor.

Original Email:
{email}

Human Feedback:
{feedback}

Update the email according to the feedback.
Return only the final email.
"""

    response = model.invoke(prompt)
    return response.content

@tool
def send_email(email: str) -> str:
    """Send email."""
    return f"""
Email sent successfully.

Final Email:

{email}
"""

# -------------------------
# Workflow
# -------------------------

topic = "Project Status"

# Step 1: Draft
draft = draft_email.invoke(
    {"topic": topic}
)

print("\nDraft Email:")
print(draft)

# Step 2: Human Review
action = input(
    "\nChoose action (approve/edit/reject): "
).lower()

if action == "approve":

    result = send_email.invoke(
        {"email": draft}
    )

    print(result)

elif action == "edit":

    feedback = input(
        "\nEnter feedback: "
    )

    updated_email = edit_email.invoke(
        {
            "email": draft,
            "feedback": feedback
        }
    )

    print("\nEdited Email:")
    print(updated_email)

    approve = input(
        "\nApprove edited email? (yes/no): "
    ).lower()

    if approve == "yes":

        result = send_email.invoke(
            {"email": updated_email}
        )

        print(result)

    else:

        print("Email not sent.")

else:

    print("Email rejected.")


Draft Email:

Subject: Meeting Request

Hello Team,

I would like to schedule a meeting regarding Project Status.

Thanks,
Dhruvi


Edited Email:
Subject: Notification of Resignation Approval

Hello Team,

I am writing to inform you that my resignation has been approved.

Thanks,
Dhruvi

Email sent successfully.

Final Email:

Subject: Notification of Resignation Approval

Hello Team,

I am writing to inform you that my resignation has been approved.

Thanks,
Dhruvi



In [65]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# -----------------------------
# Model
# -----------------------------

model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# -----------------------------
# Tool
# -----------------------------

@tool
def send_email(
    subject: str,
    body: str
) -> str:
    """Send an email."""

    return f"""
Email Sent Successfully

Subject:
{subject}

Body:
{body}
"""

# -----------------------------
# Agent
# -----------------------------

agent = create_agent(
    model=model,
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True
            }
        )
    ]
)

# -----------------------------
# Config
# -----------------------------

config = {
    "configurable": {
        "thread_id": "email-demo"
    }
}

# -----------------------------
# User Request
# -----------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":
                """
                Draft a leave approval email
                and send it.
                """
            }
        ]
    },
    config=config
)

print("Execution paused for approval")
print(response)

Execution paused for approval
{'messages': [HumanMessage(content='\n                Draft a leave approval email\n                and send it.\n                ', additional_kwargs={}, response_metadata={}, id='8b629e7e-176f-4c69-b21c-c2abac2eebb7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qrh411kc8', 'function': {'arguments': '{"body":"Dear [Manager\'s Name], I am writing to request approval for my leave from [start date] to [end date]. I have made sure to complete all my tasks and made arrangements for coverage while I am away. Please let me know if there are any issues or concerns. Thank you for considering my request. Best regards, [Your Name]","subject":"Leave Approval Email"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 227, 'total_tokens': 318, 'completion_time': 0.259717862, 'completion_tokens_details': None, 'prompt_time': 0.011867717, 'prompt_tokens_details': None, 'queue_t

In [75]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# Model
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# -----------------------------
# Tools
# -----------------------------

@tool
def draft_email(topic: str) -> str:
    """Create draft email."""

    prompt = f"""
    Write a professional email about:
    {topic}
    """

    return model.invoke(prompt).content


@tool
def edit_email(email: str, feedback: str) -> str:
    """Edit email using human feedback."""

    prompt = f"""
    Original Email:
    {email}

    Human Feedback:
    {feedback}

    Update the email based on the feedback.
    Return only the updated email.
    """

    return model.invoke(prompt).content


@tool
def send_email(email: str) -> str:
    """Send email."""

    return f"""
    Email sent successfully.

    Final Email:

    {email}
    """

# -----------------------------
# Workflow
# -----------------------------

topic = input("Enter topic: ")

# Step 1: Draft
draft = draft_email.invoke(
    {"topic": topic}
)

print("\n===== DRAFT EMAIL =====")
print(draft)

while True:

    action = input(
        "\nChoose action "
        "(approve/edit/reject): "
    ).lower()

    if action == "edit":

        feedback = input(
            "\nEnter feedback: "
        )

        draft = edit_email.invoke(
            {
                "email": draft,
                "feedback": feedback
            }
        )

        print("\n===== UPDATED EMAIL =====")
        print(draft)

    elif action == "approve":

        result = send_email.invoke(
            {"email": draft}
        )

        print("\n===== RESULT =====")
        print(result)
        break

    elif action == "reject":

        print("\nEmail rejected.")
        break

    

    else:

        print("Invalid option.")


===== DRAFT EMAIL =====
Subject: Notification of Rejection

Dear [Applicant/Client Name],

I am writing to inform you that, after careful consideration, we regret to inform you that your [application/proposal/submission] has not been successful. This decision was not made lightly, and we appreciate the time and effort you invested in [applying/submitting a proposal/submitting your work].

We recognize that this news may be disappointing, and we want to assure you that this decision is in no way a reflection on your [qualifications/abilities/potential]. Our decision is based on a thorough evaluation of all [applications/proposals/submissions] received, and we have selected the candidate whose [skills/experience/qualifications] best align with the requirements of the [position/project/opportunity].

Please know that we appreciate your interest in [our company/organization/project] and the enthusiasm you brought to the [application/submission] process. We are grateful for the opportunity